In [1]:
import mlflow
# setup to mlflow tracking server
mlflow.set_tracking_uri('http://13.51.166.199:5000')

In [2]:
# Set or create an experiment
mlflow.set_experiment("Exp. 4 - ML Algos with HP Tuning")

<Experiment: artifact_location='s3://mlfow-test60/7', creation_time=1776671282581, experiment_id='7', last_update_time=1776671282581, lifecycle_stage='active', name='Exp. 4 - ML Algos with HP Tuning', tags={}>

In [3]:
!pip install lightgbm


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import ADASYN
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

C:\Users\adina\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df = pd.read_csv('preprocessed_data.csv').dropna()
df.shape

(36662, 5)

In [7]:
# Remap the class labels from [-1, 0, 1] to [0, 1, 2]
df['category'] = df['category'].map({-1: 0, 0: 1, 1: 2})

In [8]:
# Define a function to vectorize the data using TF-IDF
def vectorize_data(X_train, X_val, X_test, max_features, ngram_range):
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
    X_train_vec = vectorizer.fit_transform(X_train['comment']).toarray()
    X_val_vec = vectorizer.transform(X_val['comment']).toarray()
    X_test_vec = vectorizer.transform(X_test['comment']).toarray()

    # Combine additional features
    X_train_combined = np.hstack([X_train_vec, X_train[['word_count', 'char_count', 'avg_word_length']].values])
    X_val_combined = np.hstack([X_val_vec, X_val[['word_count', 'char_count', 'avg_word_length']].values])
    X_test_combined = np.hstack([X_test_vec, X_test[['word_count', 'char_count', 'avg_word_length']].values])

    return X_train_combined, X_val_combined, X_test_combined

In [9]:
# Define the function that evaluates the model on validation data
def evaluate_model(model, X_val, y_val):
    y_val_pred = model.predict(X_val)  # Predict on validation set
    f1 = f1_score(y_val, y_val_pred, average='macro')  # Calculate F1 (macro)
    accuracy = accuracy_score(y_val, y_val_pred)  # Calculate accuracy
    return f1, accuracy

In [10]:
max_features = 1006
ngram_range = (1, 2)

# Split data into training, validation and testing sets
X = df[['comment', 'word_count', 'char_count', 'avg_word_length']]
y = df['category']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp)

# Vectroization and Resampling

In [11]:
# Vectorize the data
X_train_combined, X_val_combined, X_test_combined = vectorize_data(X_train, X_val, X_test, max_features, ngram_range)

In [12]:
# Apply resampling technique
X_resampled, y_resampled = ADASYN(random_state=42).fit_resample(X_train_combined, y_train)

# XGBoost

In [15]:
# Define the Optuna objective function
def objective(trial):
    # Hyperparameters to optimize
    n_estimators = trial.suggest_int("n_estimators", 50, 300, step=10)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)

    # Initialize the XGBoost model with the suggested hyperparameters
    model = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        objective='multi:softmax',
        num_class=3,  # Number of classes
        tree_method='hist',   # Use hist method for GPU
        n_jobs=2,        
        random_state=42,
    )

    # Fit the model on the resampled training data
    model.fit(X_resampled, y_resampled,eval_set=[(X_val_combined, y_val)],
    verbose=False)

    # Evaluate the model on the validation set
    f1, accuracy = evaluate_model(model, X_val_combined, y_val)

    return accuracy, f1

In [ ]:
# Run Optuna optimization
study_xgb = optuna.create_study(directions=["maximize", "maximize"], study_name="XGBoost_Optimization")  # Multi-objective optimization for both F1 and accuracy
study_xgb.optimize(objective, n_trials=100,n_jobs=4)

[I 2026-04-22 09:12:11,376] A new study created in memory with name: XGBoost_Optimization
[I 2026-04-22 09:12:51,770] Trial 2 finished with values: [0.6302964175304601, 0.5822076179713518] and parameters: {'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.02590404980173182}.
[I 2026-04-22 09:14:38,527] Trial 3 finished with values: [0.5715584651754865, 0.5323937115702795] and parameters: {'n_estimators': 180, 'max_depth': 4, 'learning_rate': 0.0007781073859374711}.
[I 2026-04-22 09:16:04,976] Trial 0 finished with values: [0.5857428623386071, 0.5482002225190422] and parameters: {'n_estimators': 250, 'max_depth': 5, 'learning_rate': 0.0006254462290790399}.
[I 2026-04-22 09:16:10,263] Trial 1 finished with values: [0.7855973813420621, 0.7628872647994077] and parameters: {'n_estimators': 250, 'max_depth': 7, 'learning_rate': 0.06424298908158592}.
[I 2026-04-22 09:16:53,260] Trial 4 finished with values: [0.7059465357337698, 0.6718257525116761] and parameters: {'n_estimators': 120, 'm